# QuSpin Integration

This notebook demonstrates how to use the QuSpin adapter to convert condensed matter
Hamiltonians into quantum circuits for resource estimation.

**Requirements:** `pip install qdk-pythonic[quspin]`

## 1. Transverse-Field Ising Model

The transverse-field Ising model is H = -J Σ Z_i Z_j - h Σ X_i.
We define it using QuSpin's operator specification format.

In [ ]:
from qdk_pythonic.adapters.quspin_adapter import (
    from_quspin_static_list,
    simulate_quspin_model,
)
from qdk_pythonic.domains.common.evolution import TrotterEvolution

L = 8
J = 1.0
h_field = 0.5

J_zz = [[J, i, (i + 1) % L] for i in range(L)]
h_x = [[h_field, i] for i in range(L)]

static_list = [["zz", J_zz], ["x", h_x]]

result = simulate_quspin_model(
    static_list=static_list,
    n_sites=L,
    time=1.0,
    trotter_steps=10,
)

print(f"Qubits:           {result['n_qubits']}")
print(f"Hamiltonian terms: {result['n_hamiltonian_terms']}")
print(f"Total gates:      {result['total_gates']}")
print(f"Circuit depth:    {result['depth']}")
print(f"Gate breakdown:   {result['gate_count']}")

## 2. Scaling Study

How do circuit resources grow with system size?

In [ ]:
print(f"{'L':>4} {'Qubits':>8} {'Terms':>8} {'Gates':>10} {'Depth':>8}")
print("-" * 42)

for L_sweep in [4, 6, 8, 10, 12, 16, 20]:
    J_zz_s = [[J, i, (i + 1) % L_sweep] for i in range(L_sweep)]
    h_x_s = [[h_field, i] for i in range(L_sweep)]
    r = simulate_quspin_model(
        static_list=[["zz", J_zz_s], ["x", h_x_s]],
        n_sites=L_sweep,
        time=1.0,
        trotter_steps=10,
    )
    print(
        f"{L_sweep:>4} {r['n_qubits']:>8} {r['n_hamiltonian_terms']:>8} "
        f"{r['total_gates']:>10} {r['depth']:>8}"
    )

## 3. Heisenberg XXZ Model

The Heisenberg model includes XX, YY, and ZZ interactions.
We define it on a 2D square lattice.

In [ ]:
Lx, Ly = 3, 3
N = Lx * Ly
Jxy = 1.0
Jz = 0.8

neighbors = []
for x in range(Lx):
    for y in range(Ly):
        site = x * Ly + y
        if x + 1 < Lx:
            neighbors.append([site, (x + 1) * Ly + y])
        if y + 1 < Ly:
            neighbors.append([site, x * Ly + (y + 1)])

J_xx = [[Jxy / 2, i, j] for i, j in neighbors]
J_yy = [[Jxy / 2, i, j] for i, j in neighbors]
J_zz = [[Jz, i, j] for i, j in neighbors]

static_xxz = [["xx", J_xx], ["yy", J_yy], ["zz", J_zz]]

result_xxz = simulate_quspin_model(
    static_list=static_xxz,
    n_sites=N,
    time=2.0,
    trotter_steps=20,
    trotter_order=2,
)

print(f"Heisenberg XXZ on {Lx}x{Ly} lattice")
print(f"  Qubits:           {result_xxz['n_qubits']}")
print(f"  Hamiltonian terms: {result_xxz['n_hamiltonian_terms']}")
print(f"  Total gates:      {result_xxz['total_gates']}")
print(f"  Circuit depth:    {result_xxz['depth']}")

## 4. Trotter Convergence

Compare first-order vs second-order Trotter at different step counts.

In [ ]:
print(f"{'Order':>6} {'Steps':>6} {'Gates':>10} {'Depth':>8}")
print("-" * 34)

for order in [1, 2]:
    for steps in [5, 10, 20, 40]:
        r = simulate_quspin_model(
            static_list=static_xxz,
            n_sites=N,
            time=2.0,
            trotter_steps=steps,
            trotter_order=order,
        )
        print(f"{order:>6} {steps:>6} {r['total_gates']:>10} {r['depth']:>8}")

## 5. Inspect Generated Q#

You can always look at the generated Q# and OpenQASM code.

In [ ]:
# Small 2-site example for readable output
small_static = [["zz", [[1.0, 0, 1]]], ["x", [[0.5, 0], [0.5, 1]]]]
hamiltonian = from_quspin_static_list(small_static, 2)
evolution = TrotterEvolution(hamiltonian, time=0.5, steps=2, order=1)
circuit = evolution.to_circuit()

print("--- Q# ---")
print(circuit.to_qsharp())
print("\n--- OpenQASM 3.0 ---")
print(circuit.to_openqasm())